# Dede Korkut sesi — kendi sesinden Piper ince ayarı
Hücreleri **sırayla** çalıştır (▶). Takıldığın yerin çıktısını Claude'a yapıştır.

Önce: *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* seç.

In [19]:
!nvidia-smi | head -15

Sat Oct 31 18:35:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1) Gerekli araçlar ve Drive

In [20]:
!pip -q install faster-whisper
!apt -qq install -y ffmpeg > /dev/null
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
# Hakan_ses.mp4 dosyasını Drive'ın ana klasörüne (MyDrive) yükle; adı farklıysa burayı değiştir
VIDEO = '/content/drive/MyDrive/Hakan_ses.mp4'
import os; print(os.path.exists(VIDEO), os.path.getsize(VIDEO)//1_000_000 if os.path.exists(VIDEO) else 0, 'MB')

True 105 MB


## 2) Konuşmayı yazıya dök ve cümlelere böl (≈15-25 dk)

In [24]:
!pip -q install nvidia-cublas-cu12 nvidia-cudnn-cu12
!rm -rf /content/UMAY
!git clone -q https://github.com/hakanuste/UMAY.git
!python3 UMAY/arac/ses_egit/hazirla.py "$VIDEO" /content/veri


347 cümle yazıldı → /content/veri/metadata.csv ve wavs/. Şimdi metadata.csv'yi gözden geçir.


In [25]:
!head -30 /content/veri/metadata.csv
!wc -l /content/veri/metadata.csv

ses_0001|Halifeliği kaldırmanın zamanı gelmişti.
ses_0002|Muhterem Efendiler, her sorunda ve her yürütüm evresinde kendisini söz konusu ettirmiş olan halifeye ve halifeliğe bir kez daha değineceğim.
ses_0003|1924 yılı başında büyük ölçüde bir ordu savaş oyunu yapılması kararlaştırılmıştı.
ses_0004|Bu savaş oyununu İzmir'de yapacaktık.
ses_0005|Bunun için 1924 yılı Ocak ayı başında İzmir'e gittim.
ses_0006|Orada iki aya yakın kaldım.
ses_0007|Oradayken halifeliğin kaldırılması zamanının geldiği yargısına varmıştım.
ses_0008|Bu işin nasıl yapıldığını kısaca anlatmaya çalışacağım.
ses_0009|Başbakan İsmet Paşa'dan 22 Ocak 1924 günlü bir kapalı tel aldım.
ses_0010|Onu olduğu gibi bilginize sunayım.
ses_0011|Türkiye Cumhurbaşkanlığı Yüksek Kapına
ses_0012|Bir süreden beri halifelik makamı ve halife ile ilgili olarak gazetelerde yok yere yanlış aşılamalara yol açabilecek saygısızca yayınlara rastlanması
ses_0013|ve özellikle ara sıra İstanbul'a giden hükümet ileri gelenlerinin ve resmi kurull

## 3) Yazıyı gözden geçir
Sol taraftaki **Dosyalar** simgesinden `veri/metadata.csv` dosyasına çift tıkla; yanlış yazılmış sözcükleri düzelt, bozuk satırları sil, kaydet. Bitince sonraki hücreyi çalıştır (Drive'a yedeklenir).

In [26]:
!mkdir -p /content/drive/MyDrive/umay_ses
!cp -r /content/veri /content/drive/MyDrive/umay_ses/
print('yedeklendi: MyDrive/umay_ses/veri')

yedeklendi: MyDrive/umay_ses/veri


## 4) Piper eğitim aracı ve Türkçe başlangıç noktası (dfki)

In [27]:
!git clone -q https://github.com/OHF-Voice/piper1-gpl.git
%cd /content/piper1-gpl
!pip -q install -e .[train]
!ls

/content/piper1-gpl
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for piper-tts (pyproject.toml) ... done
build_monotonic_align.sh  Dockerfile  MANIFEST.in     README.md  tests
CHANGELOG.md		  docs	      mypy.ini	      script	 tox.ini
CMakeLists.txt		  etc	      pylintrc	      setup.cfg
COPYING			  libpiper    pyproject.toml  setup.py
docker			  licenses    pytest.ini      src


In [31]:
!pip -q install "cython>=3.0.11" setuptools
%cd /content/piper1-gpl/src/piper/train/vits/monotonic_align
!sed -i -E 's/\) nogil:/) noexcept nogil:/' core.pyx
!grep -n "nogil" core.pyx
%cd /content/piper1-gpl
!bash build_monotonic_align.sh 2>&1 | tail -15

/content/piper1-gpl/src/piper/train/vits/monotonic_align
7:cdef void maximum_path_each(int[:,::1] path, float[:,::1] value, int t_y, int t_x, float max_neg_val=-1e9) noexcept nogil:
38:cpdef void maximum_path_c(int[:,:,::1] paths, float[:,:,::1] values, int[::1] t_ys, int[::1] t_xs) noexcept nogil:
41:  for i in prange(b, nogil=True):
/content/piper1-gpl
Compiling /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx because it changed.
[1/1] Cythonizing /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx


In [35]:
!pip install -U "cython>=3.1" setuptools 2>&1 | tail -2
import Cython; print("Cython:", Cython.__version__)
%cd /content/piper1-gpl/src/piper/train/vits/monotonic_align
!rm -f core.c *.so
%cd /content/piper1-gpl
!rm -rf build
!cat build_monotonic_align.sh
!bash build_monotonic_align.sh 2>&1 | tail -8

torch 2.11.0+cu130 requires setuptools<82, but you have setuptools 84.2.1 which is incompatible.


Cython: 3.3.0
/content/piper1-gpl/src/piper/train/vits/monotonic_align
/content/piper1-gpl
#!/usr/bin/env bash
set -eo pipefail

this_dir="$( cd "$( dirname "$0" )" && pwd )"

if [ -d "${this_dir}/.venv" ]; then
    source "${this_dir}/.venv/bin/activate"
fi

cd "${this_dir}/src/piper/train/vits/monotonic_align"
mkdir -p monotonic_align
rm -f core.c
cythonize -i core.pyx
mv core*.so monotonic_align/
Compiling /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx because it changed.
[1/1] Cythonizing /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx


In [1]:
# Bu adım derleme isterse depodaki TRAINING.md'ye bak; çıktıyı Claude'a yapıştır
!bash build_monotonic_align.sh 2>&1 | tail -5

bash: build_monotonic_align.sh: No such file or directory


In [33]:
%cd /content/piper1-gpl/src/piper/train/vits/monotonic_align
!rm -f core.c *.so
!rm -rf /content/piper1-gpl/build
!cat core.pyx

/content/piper1-gpl/src/piper/train/vits/monotonic_align
cimport cython
from cython.parallel import prange


@cython.boundscheck(False)
@cython.wraparound(False)
cdef void maximum_path_each(int[:,::1] path, float[:,::1] value, int t_y, int t_x, float max_neg_val=-1e9) noexcept nogil:
  cdef int x
  cdef int y
  cdef float v_prev
  cdef float v_cur
  cdef float tmp
  cdef int index = t_x - 1

  for y in range(t_y):
    for x in range(max(0, t_x + y - t_y), min(t_x, y + 1)):
      if x == y:
        v_cur = max_neg_val
      else:
        v_cur = value[y-1, x]
      if x == 0:
        if y == 0:
          v_prev = 0.
        else:
          v_prev = max_neg_val
      else:
        v_prev = value[y-1, x-1]
      value[y, x] += max(v_prev, v_cur)

  for y in range(t_y - 1, -1, -1):
    path[y, index] = 1
    if index != 0 and (index == y or value[y-1, index] < value[y-1, index-1]):
      index = index - 1


@cython.boundscheck(False)
@cython.wraparound(False)
cpdef void maximum_path_c(int[

In [4]:
!wc -l /content/veri/metadata.csv

347 /content/veri/metadata.csv


In [5]:
!pip -q install -U huggingface_hub
from huggingface_hub import hf_hub_download
yol = hf_hub_download(repo_id="rhasspy/piper-checkpoints", repo_type="dataset",
                      filename="tr/tr_TR/dfki/medium/epoch=5679-step=1489110.ckpt",
                      local_dir="/content/ckpt")
print(yol)
!ls -la /content/ckpt/tr/tr_TR/dfki/medium/

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


tr/tr_TR/dfki/medium/epoch=5679-step=148(…): reconstructing file:   0%|          |  0.00B /  846MB            

tr/tr_TR/dfki/medium/epoch=5679-step=148(…): downloading bytes:           |  0.00B            

/content/ckpt/tr/tr_TR/dfki/medium/epoch=5679-step=1489110.ckpt
total 826088
drwxr-xr-x 2 root root      4096 Oct 31 19:00  .
drwxr-xr-x 3 root root      4096 Oct 31 19:00  ..
-rw-r--r-- 1 root root 845898392 Oct 31 19:00 'epoch=5679-step=1489110.ckpt'


In [8]:
!echo "--- metadata satırı:"; wc -l /content/veri/metadata.csv
!echo "--- ses parçası sayısı:"; ls /content/veri/wavs | wc -l
!echo "--- monotonic_align derlemesi:"; ls /content/piper1-gpl/src/piper/train/vits/monotonic_align/*.so
!echo "--- ilk satırlar:"; head -5 /content/veri/metadata.csv

--- metadata satırı:
347 /content/veri/metadata.csv
--- ses parçası sayısı:
347
--- monotonic_align derlemesi:
ls: cannot access '/content/piper1-gpl/src/piper/train/vits/monotonic_align/*.so': No such file or directory
--- ilk satırlar:
ses_0001|Halifeliği kaldırmanın zamanı gelmişti.
ses_0002|Muhterem Efendiler, her sorunda ve her yürütüm evresinde kendisini söz konusu ettirmiş olan halifeye ve halifeliğe bir kez daha değineceğim.
ses_0003|1924 yılı başında büyük ölçüde bir ordu savaş oyunu yapılması kararlaştırılmıştı.
ses_0004|Bu savaş oyununu İzmir'de yapacaktık.
ses_0005|Bunun için 1924 yılı Ocak ayı başında İzmir'e gittim.


In [15]:
%%writefile /content/egit.py
import sys, runpy, torch
_yuk = torch.load
def _yuk2(*a, **k):
    k["weights_only"] = False
    return _yuk(*a, **k)
torch.load = _yuk2
sys.argv[0] = "piper.train"
runpy.run_module("piper.train", run_name="__main__", alter_sys=True)

Overwriting /content/egit.py


In [18]:
!apt-get -qq install -y build-essential cmake ninja-build > /dev/null
%cd /content/piper1-gpl
!python3 setup.py build_ext --inplace 2>&1 | tail -15
!bash build_monotonic_align.sh 2>&1 | tail -5
!find /content/piper1-gpl -name "espeakbridge*" -o -name "core*.so"

/content/piper1-gpl
Traceback (most recent call last):
  File "/content/piper1-gpl/setup.py", line 6, in <module>
    from skbuild import setup
ModuleNotFoundError: No module named 'skbuild'
Compiling /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx because it changed.
[1/1] Cythonizing /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx
/content/piper1-gpl/src/build/lib.linux-x86_64-cpython-313/piper/train/vits/monotonic_align/core.cpython-313-x86_64-linux-gnu.so
/content/piper1-gpl/src/piper/espeakbridge.pyi
/content/piper1-gpl/src/piper/espeakbridge.c
/content/piper1-gpl/src/piper/train/vits/monotonic_align/monotonic_align/core.cpython-313-x86_64-linux-gnu.so


In [19]:
!pip install -q scikit-build cmake ninja setuptools
%cd /content/piper1-gpl
!python3 setup.py build_ext --inplace 2>&1 | tail -20
!find /content/piper1-gpl -name "espeakbridge*.so"

/content/piper1-gpl
-- Installing: /content/piper1-gpl/_skbuild/linux-x86_64-3.13/cmake-build/espeak_ng-install/include/espeak
-- Installing: /content/piper1-gpl/_skbuild/linux-x86_64-3.13/cmake-build/espeak_ng-install/include/espeak/speak_lib.h
-- Installing: /content/piper1-gpl/_skbuild/linux-x86_64-3.13/cmake-build/espeak_ng-install/include/espeak-ng
-- Installing: /content/piper1-gpl/_skbuild/linux-x86_64-3.13/cmake-build/espeak_ng-install/include/espeak-ng/speak_lib.h
-- Installing: /content/piper1-gpl/_skbuild/linux-x86_64-3.13/cmake-build/espeak_ng-install/include/espeak-ng/encoding.h
-- Installing: /content/piper1-gpl/_skbuild/linux-x86_64-3.13/cmake-build/espeak_ng-install/include/espeak-ng/espeak_ng.h
-- Installing: /content/piper1-gpl/_skbuild/linux-x86_64-3.13/cmake-build/espeak_ng-install/bin/speak-ng
-- Installing: /content/piper1-gpl/_skbuild/linux-x86_64-3.13/cmake-build/espeak_ng-install/bin/espeak
-- Installing: /content/piper1-gpl/_skbuild/linux-x86_64-3.13/cmake-bui

## 5) İnce ayar (T4'te saatler sürer; kesilirse kaldığı yerden sürdürülür)

In [ ]:
%env TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1
import glob
CKPT = glob.glob("/content/ckpt/**/*.ckpt", recursive=True)[0]
print(CKPT)
!mkdir -p /content/drive/MyDrive/umay_ses/egitim /content/cache
!cd /content/piper1-gpl/src && python3 /content/egit.py fit \
  --data.voice_name hakan \
  --data.csv_path /content/veri/metadata.csv \
  --data.audio_dir /content/veri/wavs \
  --data.espeak_voice tr \
  --data.cache_dir /content/cache \
  --data.config_path /content/hakan.onnx.json \
  --data.batch_size 16 \
  --model.sample_rate 22050 \
  --model.warmstart_ckpt "$CKPT" \
  --trainer.max_epochs 150 \
  --trainer.default_root_dir /content/drive/MyDrive/umay_ses/egitim

env: TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1
/content/ckpt/tr/tr_TR/dfki/medium/epoch=5679-step=1489110.ckpt
/usr/local/lib/python3.13/dist-packages/lightning/fabric/utilities/seed.py:44: No seed found, seed set to 0
Seed set to 0
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:piper.train.vits.dataset:Processing utterances...
INFO:piper.train.vits.dataset:Processed 347 utterance(s)
I0000 00:00:1793474605.587735   19501 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU i

## 6) ONNX'e çevir ve Drive'a koy

In [ ]:
import glob
son = sorted(glob.glob('/content/**/*.ckpt', recursive=True), key=os.path.getmtime)[-1]
print(son)
!python3 -m piper.train.export_onnx --checkpoint "$son" --output-file /content/hakan.onnx
!cp /content/hakan.onnx /content/drive/MyDrive/umay_ses/hakan.onnx
!cp /content/cikti/config.json /content/drive/MyDrive/umay_ses/hakan.onnx.json

Mac'te: `hakan.onnx` ve `hakan.onnx.json` dosyalarını `~/.local/share/umay/ses/` içine koy; uygulamada Ayarlar → Ses modeli.